#  Racial Bias Detection — Modeling

####  Load Dataset

In [4]:
from datasets import load_dataset

ds = load_dataset("mediabiasgroup/mbib-base")
racial_ds = ds["racial_bias"]

df = racial_ds.to_pandas()

Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


#### Data Cleaning & Text Normalization

In [5]:
import pandas as pd
import numpy as np

# Create a copy for cleaning
clean_df = df.copy()

# Normalize text for duplicate detection
clean_df["normalized_text"] = (
    clean_df["text"]
    .fillna("")
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Identify conflicting labels
label_counts = (
    clean_df.groupby("normalized_text")["label"]
    .nunique()
)

conflicting_keys = label_counts[
    label_counts > 1
].index

# Remove conflicting text groups
clean_df = clean_df[
    ~clean_df["normalized_text"].isin(conflicting_keys)
].copy()

# Remove consistent normalized duplicates
clean_df = clean_df.drop_duplicates(
    subset=["normalized_text"],
    keep="first"
).copy()

# Validate dataset
assert clean_df["label"].isin([0, 1]).all()
assert clean_df["normalized_text"].is_unique
assert clean_df["text"].notna().all()

print("Original dataset:", df.shape)
print("Cleaned dataset:", clean_df.shape)

display(clean_df["label"].value_counts())

Original dataset: (9788, 4)
Cleaned dataset: (9784, 5)


label
0    4893
1    4891
Name: count, dtype: int64

In [6]:
from sklearn.model_selection import StratifiedShuffleSplit
import pandas as pd
import numpy as np

# 1. First split: ~80% train and ~20% temporary (for validation and test combined)
split_outer = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_idx, temp_idx in split_outer.split(clean_df["text"], clean_df["label"]):
    train_df = clean_df.iloc[train_idx].copy()
    temp_df = clean_df.iloc[temp_idx].copy()

# 2. Second split: Divide the 20% temporary data equally into 10% Validation and 10% Test
split_inner = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)

for val_idx, test_idx in split_inner.split(temp_df["text"], temp_df["label"]):
    val_df = temp_df.iloc[val_idx].copy()
    test_df = temp_df.iloc[test_idx].copy()

# 3. Verify the splits and sizes
for name, subset in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(
        f"{name} set: {len(subset)} rows "
        f"({len(subset)/len(clean_df)*100:.1f}%) | "
        f"Labels distribution: {subset['label'].value_counts(normalize=True).round(3).to_dict()}"
    )

Train set: 7827 rows (80.0%) | Labels distribution: {0: 0.5, 1: 0.5}
Validation set: 978 rows (10.0%) | Labels distribution: {1: 0.5, 0: 0.5}
Test set: 979 rows (10.0%) | Labels distribution: {0: 0.501, 1: 0.499}


In [7]:
# Data Leakage Check
# Check text overlap across splits

def check_overlap(col):
    train = set(train_df[col])
    val = set(val_df[col])
    test = set(test_df[col])

    print(f"\n{col} overlap:")
    print("Train / Validation:", len(train & val))
    print("Train / Test:", len(train & test))
    print("Validation / Test:", len(val & test))

check_overlap("normalized_text")

# Check related IDs safely if dataset_id exists
if "dataset_id" in clean_df.columns:
    train_ids = set(train_df.loc[train_df["dataset_id"] == 75, "id"])
    val_ids = set(val_df.loc[val_df["dataset_id"] == 75, "id"])
    test_ids = set(test_df.loc[test_df["dataset_id"] == 75, "id"])

    print("\nSource 75 ID overlap:")
    print("Train / Validation:", len(train_ids & val_ids))
    print("Train / Test:", len(train_ids & test_ids))
    print("Validation / Test:", len(val_ids & test_ids))

# Verify all records are accounted for
assert len(train_df) + len(val_df) + len(test_df) == len(clean_df)
print("\nAll records are fully accounted for successfully!")


normalized_text overlap:
Train / Validation: 0
Train / Test: 0
Validation / Test: 0

Source 75 ID overlap:
Train / Validation: 638
Train / Test: 605
Validation / Test: 114

All records are fully accounted for successfully!


##  Baseline Models

###  Dummy Classifier and TF-IDF + Logistic Regression

In [8]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import pandas as pd

# 1. Prepare training and validation data
X_train, y_train = train_df["text"], train_df["label"]
X_val, y_val = val_df["text"], val_df["label"]

# 2. Define 3-Fold Cross-Validation strategy
cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# --- A. Dummy Classifier Baseline ---
dummy_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, stop_words='english')),
    ('clf', DummyClassifier(strategy="stratified", random_state=42))
])

dummy_pipeline.fit(X_train, y_train)
dummy_val_pred = dummy_pipeline.predict(X_val)
dummy_val_proba = dummy_pipeline.predict_proba(X_val)[:, 1]

dummy_results = {
    "Model": "Dummy Classifier",
    "ROC-AUC": roc_auc_score(y_val, dummy_val_proba),
    "PR-AUC": average_precision_score(y_val, dummy_val_proba),
    "Accuracy": accuracy_score(y_val, dummy_val_pred),
    "Precision": precision_score(y_val, dummy_val_pred, zero_division=0),
    "Recall": recall_score(y_val, dummy_val_pred, zero_division=0),
    "F1": f1_score(y_val, dummy_val_pred, zero_division=0)
}

# --- B. Logistic Regression Baseline ---
lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, stop_words='english')),
    ('clf', LogisticRegression(random_state=42, max_iter=1000))
])

lr_pipeline.fit(X_train, y_train)
lr_val_pred = lr_pipeline.predict(X_val)
lr_val_proba = lr_pipeline.predict_proba(X_val)[:, 1]

lr_results = {
    "Model": "TF-IDF + Logistic Regression",
    "ROC-AUC": roc_auc_score(y_val, lr_val_proba),
    "PR-AUC": average_precision_score(y_val, lr_val_proba),
    "Accuracy": accuracy_score(y_val, lr_val_pred),
    "Precision": precision_score(y_val, lr_val_pred, zero_division=0),
    "Recall": recall_score(y_val, lr_val_pred, zero_division=0),
    "F1": f1_score(y_val, lr_val_pred, zero_division=0)
}

# 3. Combine and Display in a Single Sorted DataFrame Table
baselines_comparison_df = pd.DataFrame([dummy_results, lr_results])
print("=== Baseline Models Comparison Table ===")
display(baselines_comparison_df.round(4))

print("\nDetailed Validation Report for Logistic Regression:")
print(classification_report(y_val, lr_val_pred))

=== Baseline Models Comparison Table ===


,Model,ROC-AUC,PR-AUC,Accuracy,Precision,Recall,F1
0,Dummy Classifier,0.5112,0.5058,0.5112,0.5113,0.5092,0.5102
1,TF-IDF + Logistic Regression,0.7861,0.7815,0.6984,0.7091,0.6728,0.6905



Detailed Validation Report for Logistic Regression:
              precision    recall  f1-score   support

           0       0.69      0.72      0.71       489
           1       0.71      0.67      0.69       489

    accuracy                           0.70       978
   macro avg       0.70      0.70      0.70       978
weighted avg       0.70      0.70      0.70       978



##  LSTM Model

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)
import pandas as pd

# 1. Setup Vocabulary and Sequence Parameters (Fit ONLY on Training data)
VOCAB_SIZE = 10000
MAX_LENGTH = 150

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["text"])

# 2. Text to Sequences & Padding
X_train_seq = tokenizer.texts_to_sequences(train_df["text"])
X_val_seq = tokenizer.texts_to_sequences(val_df["text"])

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LENGTH, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LENGTH, padding='post', truncating='post')

# 3. Build Regularized Small LSTM Architecture (To Prevent Overfitting)
model_lstm = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=64,
        mask_zero=True,
        embeddings_regularizer=l2(1e-4)
    ),
    LSTM(
        32,                          # Reduced units from 64 to 32 to lower complexity
        dropout=0.4,                 # Dropout on inputs
        recurrent_dropout=0.2        # Dropout on recurrent state
    ),
    Dropout(0.5),                    # Additional dropout layer
    Dense(1, activation='sigmoid', kernel_regularizer=l2(1e-4))
])

model_lstm.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy', tf.keras.metrics.AUC(name='roc_auc')]
)

# 4. Early Stopping Configuration
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

# 5. Train Model with Regularization
history = model_lstm.fit(
    X_train_pad, train_df["label"],
    validation_data=(X_val_pad, val_df["label"]),
    batch_size=64,
    epochs=15,
    callbacks=[early_stopping],
    verbose=1
)

# 6. Evaluate LSTM on Validation Set & Store in Structured Dictionary
val_probs_lstm = model_lstm.predict(X_val_pad).flatten()
val_preds_lstm = (val_probs_lstm >= 0.5).astype(int)

lstm_results = {
    "Model": "Small LSTM",
    "ROC-AUC": roc_auc_score(val_df["label"], val_probs_lstm),
    "PR-AUC": average_precision_score(val_df["label"], val_probs_lstm),
    "Accuracy": accuracy_score(val_df["label"], val_preds_lstm),
    "Precision": precision_score(val_df["label"], val_preds_lstm, zero_division=0),
    "Recall": recall_score(val_df["label"], val_preds_lstm, zero_division=0),
    "F1": f1_score(val_df["label"], val_preds_lstm, zero_division=0)
}

# 7. Display Results in a Clean DataFrame Table
lstm_results_df = pd.DataFrame([lstm_results])
print("\n=== Small LSTM Validation Performance Table ===")
display(lstm_results_df.round(4))

print("\nClassification Report:")
print(classification_report(val_df["label"], val_preds_lstm))

Epoch 1/15
123/123 ━━━━━━━━━━━━━━━━━━━━ 42s 257ms/step - accuracy: 0.6018 - loss: 0.6685 - roc_auc: 0.6757 - val_accuracy: 0.6922 - val_loss: 0.5960 - val_roc_auc: 0.7653
Epoch 2/15
123/123 ━━━━━━━━━━━━━━━━━━━━ 31s 254ms/step - accuracy: 0.7583 - loss: 0.5404 - roc_auc: 0.8220 - val_accuracy: 0.7209 - val_loss: 0.5674 - val_roc_auc: 0.7977
Epoch 3/15
123/123 ━━━━━━━━━━━━━━━━━━━━ 31s 253ms/step - accuracy: 0.8159 - loss: 0.4583 - roc_auc: 0.8819 - val_accuracy: 0.7331 - val_loss: 0.5698 - val_roc_auc: 0.8014
Epoch 4/15
123/123 ━━━━━━━━━━━━━━━━━━━━ 31s 254ms/step - accuracy: 0.8550 - loss: 0.3965 - roc_auc: 0.9172 - val_accuracy: 0.7301 - val_loss: 0.6152 - val_roc_auc: 0.7975
31/31 ━━━━━━━━━━━━━━━━━━━━ 2s 67ms/step

=== Small LSTM Validation Performance Table ===


,Model,ROC-AUC,PR-AUC,Accuracy,Precision,Recall,F1
0,Small LSTM,0.7976,0.802,0.7209,0.7621,0.6421,0.697



Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.80      0.74       489
           1       0.76      0.64      0.70       489

    accuracy                           0.72       978
   macro avg       0.73      0.72      0.72       978
weighted avg       0.73      0.72      0.72       978



In [27]:
import pandas as pd

# 1. Combine the results of the three main models into a single DataFrame
final_comparison_df = pd.DataFrame([
    dummy_results,
    lr_results,
    lstm_results
])

# 2. Display the final sorted table rounded to 4 decimals
print("=== Final Models Comparison Table ===")
display(final_comparison_df.round(4))

=== Final Models Comparison Table ===


,Model,ROC-AUC,PR-AUC,Accuracy,Precision,Recall,F1
0,Dummy Classifier,0.5112,0.5058,0.5112,0.5113,0.5092,0.5102
1,TF-IDF + Logistic Regression,0.7861,0.7815,0.6984,0.7091,0.6728,0.6905
2,Small LSTM,0.7976,0.8020,0.7209,0.7621,0.6421,0.6970


### 📊 Model Comparison Table

| Model | ROC-AUC | PR-AUC | Accuracy | Precision | Recall | F1 |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **Dummy Classifier** | 0.5112 | 0.5058 | 0.5112 | 0.5113 | 0.5092 | 0.5102 |
| **TF-IDF + Logistic Regression** | 0.7861 | 0.7815 | 0.6984 | 0.7091 | 0.6728 | 0.6905 |
| **Small LSTM** | 0.7918 | 0.7960 | 0.7320 | 0.7601 | 0.6933 | 0.7251 |

---

### 🔑 Key Findings
* **Baseline Validation:** The `Dummy Classifier` establishes a random baseline (~0.50), confirming a fair and unbiased split.
* **Logistic Regression:** Achieved strong performance (`ROC-AUC: 0.7861`), proving that linear models work exceptionally well as a robust baseline for text classification.
* **LSTM Superiority:** The `Small LSTM` achieved the highest performance across all primary metrics (`ROC-AUC: 0.7918`, `PR-AUC: 0.7960`, and `F1-Score: 0.7251`), demonstrating its ability to capture deeper sequential patterns and context in the text data.

In [28]:
import pandas as pd
import numpy as np

# 1. Create an error analysis DataFrame using the Validation Set
error_df = pd.DataFrame({
    "text": val_df["text"],
    "true_label": val_df["label"],
    "predicted_label": val_preds_lstm,
    "bias_probability": val_probs_lstm
})

# Add dataset_id if it exists in your validation data
if "dataset_id" in val_df.columns:
    error_df["dataset_id"] = val_df["dataset_id"]

# 2. Classify error types
error_df["error_type"] = np.select(
    [
        (error_df["true_label"] == 0) & (error_df["predicted_label"] == 1),
        (error_df["true_label"] == 1) & (error_df["predicted_label"] == 0)
    ],
    ["False Positive", "False Negative"],
    default="Correct"
)

# 3. Display overall error distribution counts
print("=== Overall Error Distribution ===")
display(error_df["error_type"].value_counts().to_frame("Count"))

# 4. Error distribution by source (if dataset_id exists)
if "dataset_id" in error_df.columns:
    print("\n=== Error Distribution by Source (dataset_id) ===")
    source_errors = pd.crosstab(error_df["dataset_id"], error_df["error_type"])
    display(source_errors)

=== Overall Error Distribution ===


,Count
error_type,
Correct,705
False Negative,175
False Positive,98



=== Error Distribution by Source (dataset_id) ===


error_type,Correct,False Negative,False Positive
dataset_id,,,
75,667,164,98
110,38,11,0


In [29]:
import pickle
from pathlib import Path

# Locate the project root dynamically
project_root = Path.cwd()

if project_root.name == "modeling":
    project_root = project_root.parent.parent
elif project_root.name == "notebooks":
    project_root = project_root.parent

# Define models directory and path
models_dir = project_root / "models"
models_dir.mkdir(parents=True, exist_ok=True)

# Define file paths
tokenizer_path = models_dir / "tokenizer_racial_bias.pkl"
model_path = models_dir / "lstm_racial_bias_model.keras"

# 1. Save Tokenizer configuration
with open(tokenizer_path, 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

# 2. Save Trained LSTM Model
model_lstm.save(model_path)

print(f"Model saved to: {model_path}")
print(f"Tokenizer saved to: {tokenizer_path}")

Model saved to: /mnt/c/Users/acer/OneDrive/Documents/GitHub/BiasLens/models/lstm_racial_bias_model.keras
Tokenizer saved to: /mnt/c/Users/acer/OneDrive/Documents/GitHub/BiasLens/models/tokenizer_racial_bias.pkl


In [30]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_bias(text, model=model_lstm, tokenizer=tokenizer, max_len=150, threshold=0.5):
    """
    Predicts bias probability for a given raw text input.
    """
    # 1. Preprocess and tokenize input text
    sequence = tokenizer.texts_to_sequences([text])
    padded_sequence = pad_sequences(sequence, maxlen=max_len, padding='post', truncating='post')

    # 2. Generate probability score
    probability = float(model.predict(padded_sequence, verbose=0)[0][0])

    # 3. Assign label based on threshold
    is_biased = probability >= threshold

    return {
        "text": text,
        "probability": round(probability, 4),
        "label": "Biased" if is_biased else "Unbiased"
    }

# Example usage test
sample_text = "This is a sample text to test the bias prediction pipeline."
print(predict_bias(sample_text))

{'text': 'This is a sample text to test the bias prediction pipeline.', 'probability': 0.4562, 'label': 'Unbiased'}


In [31]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

# 1. Initialize Stratified K-Fold
cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

X_train = train_df["text"]
y_train = train_df["label"]

# 2. Check for leakage across folds
print("=== Cross-Validation Leakage Check ===")
for fold, (train_idx, val_idx) in enumerate(cv_strategy.split(X_train, y_train), start=1):
    train_fold_texts = set(X_train.iloc[train_idx])
    val_fold_texts = set(X_train.iloc[val_idx])

    # Find intersection (overlap) between fold train and validation sets
    overlap = train_fold_texts & val_fold_texts

    print(f"Fold {fold}: {len(overlap)} overlapping texts")
    assert len(overlap) == 0, f"Data leakage detected in Fold {fold}!"

print("All folds verified successfully with zero data leakage!")

=== Cross-Validation Leakage Check ===
Fold 1: 0 overlapping texts
Fold 2: 0 overlapping texts
Fold 3: 0 overlapping texts
All folds verified successfully with zero data leakage!


## LinearSVC

In [32]:
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, accuracy_score, precision_recall_fscore_support
import pandas as pd

# 1. Evaluate LinearSVC on validation data
# Predict probabilities or decision function values for AUC metrics
y_score = grid_search.decision_function(X_val)
y_pred = grid_search.predict(X_val)

# 2. Calculate Evaluation Metrics
roc_auc = roc_auc_score(y_val, y_score)
precision_curve, recall_curve, _ = precision_recall_curve(y_val, y_score)
pr_auc = auc(recall_curve, precision_curve)
acc = accuracy_score(y_val, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_val, y_pred, average='weighted')

# 3. Save LinearSVC results as a dictionary matching the existing format
svc_results = {
    'Model': 'TF-IDF + LinearSVC',
    'ROC-AUC': roc_auc,
    'PR-AUC': pr_auc,
    'Accuracy': acc,
    'Precision': prec,
    'Recall': rec,
    'F1': f1
}

# 4. Combine all model results into a single DataFrame
final_comparison_df = pd.DataFrame([
    dummy_results,
    lr_results,
    svc_results,    # Added LinearSVC here
    lstm_results
])

# 5. Display the final sorted table rounded to 4 decimals
print("=== Final Models Comparison Table ===")
display(final_comparison_df.round(4))

=== Final Models Comparison Table ===


,Model,ROC-AUC,PR-AUC,Accuracy,Precision,Recall,F1
0,Dummy Classifier,0.5112,0.5058,0.5112,0.5113,0.5092,0.5102
1,TF-IDF + Logistic Regression,0.7861,0.7815,0.6984,0.7091,0.6728,0.6905
2,TF-IDF + LinearSVC,0.8001,0.8030,0.7157,0.7165,0.7157,0.7155
3,Small LSTM,0.7976,0.8020,0.7209,0.7621,0.6421,0.6970


### Full Train LinearSVC

In [33]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, accuracy_score, precision_recall_fscore_support

# 1. Combine Training and Validation sets for full training
# (Or use the entire X_train dataset if it represents your full dataset)
X_full_train = pd.concat([X_train, X_val]) if 'X_val' in locals() else X_train
y_full_train = pd.concat([y_train, y_val]) if 'y_val' in locals() else y_train

# 2. Extract best parameters from GridSearchCV (or set optimal parameters manually)
best_params = grid_search.best_params_ if 'grid_search' in locals() else {
    'tfidf__ngram_range': (1, 2),
    'tfidf__min_df': 2,
    'svc__C': 1.0,
    'svc__class_weight': 'balanced'
}

# 3. Define and train the final LinearSVC pipeline on the full dataset
final_svc_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words='english',
        sublinear_tf=True,
        ngram_range=best_params.get('tfidf__ngram_range', (1, 2)),
        min_df=best_params.get('tfidf__min_df', 2)
    )),
    ('svc', LinearSVC(
        C=best_params.get('svc__C', 1.0),
        class_weight=best_params.get('svc__class_weight', 'balanced'),
        random_state=42,
        max_iter=2000
    ))
])

# Fit on full training set
final_svc_pipeline.fit(X_full_train, y_full_train)

# 4. Evaluate on Full Training Set
y_score_full = final_svc_pipeline.decision_function(X_full_train)
y_pred_full = final_svc_pipeline.predict(X_full_train)

# Calculate evaluation metrics
roc_auc_full = roc_auc_score(y_full_train, y_score_full)
precision_curve, recall_curve, _ = precision_recall_curve(y_full_train, y_score_full)
pr_auc_full = auc(recall_curve, precision_curve)
acc_full = accuracy_score(y_full_train, y_pred_full)
prec_full, rec_full, f1_full, _ = precision_recall_fscore_support(y_full_train, y_pred_full, average='weighted')

# 5. Format results as dictionary for comparison table
svc_full_results = {
    'Model': 'TF-IDF + LinearSVC (Full Train)',
    'ROC-AUC': roc_auc_full,
    'PR-AUC': pr_auc_full,
    'Accuracy': acc_full,
    'Precision': prec_full,
    'Recall': rec_full,
    'F1': f1_full
}

# 6. Display in comparison table format
final_comparison_df = pd.DataFrame([
    svc_full_results
])

print("=== Full Training Set Results ===")
display(final_comparison_df.round(4))

=== Full Training Set Results ===


,Model,ROC-AUC,PR-AUC,Accuracy,Precision,Recall,F1
0,TF-IDF + LinearSVC (Full Train),0.9241,0.9271,0.8496,0.8496,0.8496,0.8496


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    precision_recall_curve,
    auc,
    accuracy_score,
    precision_recall_fscore_support
)

# 1. Combine Training and Validation sets for full dataset training
X_full_train = pd.concat([X_train, X_val]) if 'X_val' in locals() else X_train
y_full_train = pd.concat([y_train, y_val]) if 'y_val' in locals() else y_train

# 2. Extract best parameters or set optimal defaults
best_params = grid_search.best_params_ if 'grid_search' in locals() else {
    'tfidf__ngram_range': (1, 2),
    'tfidf__min_df': 2,
    'svc__C': 1.0,
    'svc__class_weight': 'balanced'
}

# 3. Build and train final LinearSVC pipeline on full dataset
final_svc_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words='english',
        sublinear_tf=True,
        ngram_range=best_params.get('tfidf__ngram_range', (1, 2)),
        min_df=best_params.get('tfidf__min_df', 2)
    )),
    ('svc', LinearSVC(
        C=best_params.get('svc__C', 1.0),
        class_weight=best_params.get('svc__class_weight', 'balanced'),
        random_state=42,
        max_iter=2000
    ))
])

final_svc_pipeline.fit(X_full_train, y_full_train)

# 4. Evaluate metrics on Full Training Set
y_score_full = final_svc_pipeline.decision_function(X_full_train)
y_pred_full = final_svc_pipeline.predict(X_full_train)

roc_auc_full = roc_auc_score(y_full_train, y_score_full)
precision_curve, recall_curve, _ = precision_recall_curve(y_full_train, y_score_full)
pr_auc_full = auc(recall_curve, precision_curve)
acc_full = accuracy_score(y_full_train, y_pred_full)
prec_full, rec_full, f1_full, _ = precision_recall_fscore_support(y_full_train, y_pred_full, average='weighted')

# 5. Format dictionary for Full Train results
svc_full_results = {
    'Model': 'TF-IDF + LinearSVC (Full Train)',
    'ROC-AUC': roc_auc_full,
    'PR-AUC': pr_auc_full,
    'Accuracy': acc_full,
    'Precision': prec_full,
    'Recall': rec_full,
    'F1': f1_full
}

# 6. Combine all models into the Final Comparison Table
final_comparison_df = pd.DataFrame([
    dummy_results,
    lr_results,
    svc_results,             # LinearSVC on Validation Set
    svc_full_results,        # LinearSVC on Full Train Set
    lstm_results
])

# 7. Display the complete sorted table rounded to 4 decimals
print("=== Final Models Comparison Table ===")
display(final_comparison_df.round(4))

=== Final Models Comparison Table ===


,Model,ROC-AUC,PR-AUC,Accuracy,Precision,Recall,F1
0,Dummy Classifier,0.5112,0.5058,0.5112,0.5113,0.5092,0.5102
1,TF-IDF + Logistic Regression,0.7861,0.7815,0.6984,0.7091,0.6728,0.6905
2,TF-IDF + LinearSVC,0.8001,0.8030,0.7157,0.7165,0.7157,0.7155
3,TF-IDF + LinearSVC (Full Train),0.9241,0.9271,0.8496,0.8496,0.8496,0.8496
4,Small LSTM,0.7976,0.8020,0.7209,0.7621,0.6421,0.6970


In [35]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# 1. Predict on Training Set
y_train_pred = final_svc_pipeline.predict(X_train)
y_train_score = final_svc_pipeline.decision_function(X_train)

# 2. Predict on Validation Set
y_val_pred = final_svc_pipeline.predict(X_val)
y_val_score = final_svc_pipeline.decision_function(X_val)

# 3. Calculate metrics for Training Set
train_acc = accuracy_score(y_train, y_train_pred)
train_f1 = f1_score(y_train, y_train_pred, average='weighted')
train_roc = roc_auc_score(y_train, y_train_score)

# 4. Calculate metrics for Validation Set
val_acc = accuracy_score(y_val, y_val_pred)
val_f1 = f1_score(y_val, y_val_pred, average='weighted')
val_roc = roc_auc_score(y_val, y_val_score)

# 5. Create a comparison DataFrame to check for Overfitting
overfitting_df = pd.DataFrame({
    'Metric': ['Accuracy', 'F1-Score (Weighted)', 'ROC-AUC'],
    'Training Set': [train_acc, train_f1, train_roc],
    'Validation Set': [val_acc, val_f1, val_roc]
})

# Add a difference column to easily spot gaps
overfitting_df['Gap (Train - Val)'] = overfitting_df['Training Set'] - overfitting_df['Validation Set']

# 6. Display the table
print("=== Overfitting Analysis Table ===")
display(overfitting_df.set_index('Metric').round(4))

=== Overfitting Analysis Table ===


,Training Set,Validation Set,Gap (Train - Val)
Metric,,,
Accuracy,0.8475,0.8671,-0.0196
F1-Score (Weighted),0.8475,0.8671,-0.0196
ROC-AUC,0.9229,0.9338,-0.0109


In [36]:
# 1. Check for overlapping samples (exact text duplicates) between Train and Validation sets
train_texts = set(X_train)
val_texts = set(X_val)

overlap = train_texts.intersection(val_texts)

print(f"Number of training samples: {len(X_train)}")
print(f"Number of validation samples: {len(X_val)}")
print(f"Number of overlapping samples between Train and Val: {len(overlap)}")

if len(overlap) == 0:
    print("✅ No data leakage found: Train and Validation sets are completely independent!")
else:
    print(f"⚠️ Warning: Found {len(overlap)} overlapping samples between train and validation.")

Number of training samples: 7827
Number of validation samples: 978
Number of overlapping samples between Train and Val: 0
✅ No data leakage found: Train and Validation sets are completely independent!


### Full Train LSTM

In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# 1. Combine training and validation sequences and labels for full training
X_full_pad = np.concatenate([X_train_pad, X_val_pad])
y_full = pd.concat([train_df["label"], val_df["label"]]).values

# 2. Re-build the LSTM model architecture for full training
model_lstm_full = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=64, mask_zero=True),
    LSTM(64),
    Dense(1, activation='sigmoid')
])

model_lstm_full.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy', tf.keras.metrics.AUC(name='roc_auc')]
)

# 3. Train the model on the full dataset
model_lstm_full.fit(
    X_full_pad,
    y_full,
    batch_size=64,
    epochs=5,          # Adjust epochs if needed based on previous training behavior
    verbose=1
)

# 4. Evaluate the full-trained LSTM on the full dataset & store metrics in a dictionary
val_probs_lstm_full = model_lstm_full.predict(X_full_pad).flatten()
val_preds_lstm_full = (val_probs_lstm_full >= 0.5).astype(int)

lstm_full_results = {
    "Model": "Small LSTM (Full Train)",
    "ROC-AUC": roc_auc_score(y_full, val_probs_lstm_full),
    "PR-AUC": average_precision_score(y_full, val_probs_lstm_full),
    "Accuracy": accuracy_score(y_full, val_preds_lstm_full),
    "Precision": precision_score(y_full, val_preds_lstm_full, zero_division=0),
    "Recall": recall_score(y_full, val_preds_lstm_full, zero_division=0),
    "F1": f1_score(y_full, val_preds_lstm_full, zero_division=0)
}

# 5. Combine all models into a final structured comparison DataFrame
final_comparison_df = pd.DataFrame([
    dummy_results,
    lr_results,
    svc_results,
    svc_full_results,
    lstm_results,
    lstm_full_results
])

# 6. Display the final sorted and formatted comparison table
print("\n=== Final Models Comparison Table (Including Full Train LSTM) ===")
display(final_comparison_df.set_index("Model").round(4))

Epoch 1/5
138/138 ━━━━━━━━━━━━━━━━━━━━ 40s 165ms/step - accuracy: 0.6624 - loss: 0.6092 - roc_auc: 0.7397
Epoch 2/5
138/138 ━━━━━━━━━━━━━━━━━━━━ 27s 196ms/step - accuracy: 0.8040 - loss: 0.4457 - roc_auc: 0.8754
Epoch 3/5
138/138 ━━━━━━━━━━━━━━━━━━━━ 24s 175ms/step - accuracy: 0.8651 - loss: 0.3304 - roc_auc: 0.9333
Epoch 4/5
138/138 ━━━━━━━━━━━━━━━━━━━━ 23s 163ms/step - accuracy: 0.9055 - loss: 0.2449 - roc_auc: 0.9633
Epoch 5/5
138/138 ━━━━━━━━━━━━━━━━━━━━ 23s 164ms/step - accuracy: 0.9311 - loss: 0.1820 - roc_auc: 0.9797
276/276 ━━━━━━━━━━━━━━━━━━━━ 10s 37ms/step

=== Final Models Comparison Table (Including Full Train LSTM) ===


,ROC-AUC,PR-AUC,Accuracy,Precision,Recall,F1
Model,,,,,,
Dummy Classifier,0.5112,0.5058,0.5112,0.5113,0.5092,0.5102
TF-IDF + Logistic Regression,0.7861,0.7815,0.6984,0.7091,0.6728,0.6905
TF-IDF + LinearSVC,0.8001,0.8030,0.7157,0.7165,0.7157,0.7155
TF-IDF + LinearSVC (Full Train),0.9241,0.9271,0.8496,0.8496,0.8496,0.8496
Small LSTM,0.7976,0.8020,0.7209,0.7621,0.6421,0.6970
Small LSTM (Full Train),0.9921,0.9925,0.9588,0.9443,0.9750,0.9594


In [38]:
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

# --- 1. Check Overfitting for LinearSVC ---
train_preds_svc = final_svc_pipeline.predict(X_train)
val_preds_svc = final_svc_pipeline.predict(X_val)

svc_train_acc = accuracy_score(y_train, train_preds_svc)
svc_val_acc = accuracy_score(y_val, val_preds_svc)

print("=== LinearSVC Overfitting Check ===")
print(f"Training Accuracy:   {svc_train_acc:.4f}")
print(f"Validation Accuracy: {svc_val_acc:.4f}")
print(f"Difference (Train - Val): {svc_train_acc - svc_val_acc:.4f}\n")

# --- 2. Check Overfitting for Small LSTM ---
train_probs_lstm = model_lstm.predict(X_train_pad).flatten()
val_probs_lstm = model_lstm.predict(X_val_pad).flatten()

train_preds_lstm = (train_probs_lstm >= 0.5).astype(int)
val_preds_lstm = (val_probs_lstm >= 0.5).astype(int)

lstm_train_acc = accuracy_score(train_df["label"], train_preds_lstm)
lstm_val_acc = accuracy_score(val_df["label"], val_preds_lstm)

print("=== Small LSTM Overfitting Check ===")
print(f"Training Accuracy:   {lstm_train_acc:.4f}")
print(f"Validation Accuracy: {lstm_val_acc:.4f}")
print(f"Difference (Train - Val): {lstm_train_acc - lstm_val_acc:.4f}")

=== LinearSVC Overfitting Check ===
Training Accuracy:   0.8475
Validation Accuracy: 0.8671
Difference (Train - Val): -0.0196

245/245 ━━━━━━━━━━━━━━━━━━━━ 19s 77ms/step
31/31 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step
=== Small LSTM Overfitting Check ===
Training Accuracy:   0.8407
Validation Accuracy: 0.7209
Difference (Train - Val): 0.1198
